In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install -q datasets scikit-learn

In [4]:
# !pip install -q datasets scikit-learn
import json, os, io
import numpy as np
import pandas as pd

OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)

REQUIRED = ["air_temp", "process_temp", "rpm", "torque", "tool_wear", "failure"]

def normalize_columns(df):
    """Map the various column namings of AI4I 2020 to a standard schema."""
    colmap = {}
    for c in df.columns:
        lc = c.lower()
        if "air temperature" in lc:      colmap[c] = "air_temp"
        elif "process temperature" in lc: colmap[c] = "process_temp"
        elif "rotational speed" in lc:    colmap[c] = "rpm"
        elif "torque" in lc:              colmap[c] = "torque"
        elif "tool wear" in lc:           colmap[c] = "tool_wear"
        elif lc in ("machine failure", "target", "failure"): colmap[c] = "failure"
        elif lc == "type":                colmap[c] = "type"
    df = df.rename(columns=colmap)
    return df

def try_huggingface():
    """AI4I 2020 is mirrored on the HF Hub under several dataset ids; try a
    few known candidates. Any working mirror with the right columns is fine."""
    try:
        from datasets import load_dataset
    except ImportError:
        print("  'datasets' library not installed — skipping HF (run Cell 1)")
        return None
    candidates = [
        "pauhidalgoo/ai4i2020",
        "mstz/machine_failure",
        "Chances/ai4i2020",
    ]
    for ds_id in candidates:
        try:
            ds = load_dataset(ds_id, split="train")
            df = normalize_columns(ds.to_pandas())
            if all(c in df.columns for c in REQUIRED):
                print(f"Loaded from Hugging Face: {ds_id} ({len(df)} rows)")
                return df
        except Exception as e:
            print(f"  HF candidate {ds_id} failed: {type(e).__name__}")
    return None

def try_uci():
    """Original source: UCI ML repository (public CSV)."""
    import urllib.request
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"
    try:
        with urllib.request.urlopen(url, timeout=30) as r:
            df = normalize_columns(pd.read_csv(io.BytesIO(r.read())))
        if all(c in df.columns for c in REQUIRED):
            print(f"Loaded from UCI ({len(df)} rows)")
            return df
    except Exception as e:
        print(f"  UCI fallback failed: {type(e).__name__}")
    return None

def synthetic():
    """Last-resort simulator with the same schema & similar physics so the
    rest of the pipeline is identical. Clearly labeled as synthetic."""
    print("Using SYNTHETIC sensor data (both downloads failed).")
    n = 10000; rng = np.random.default_rng(0)
    air = 300 + rng.normal(0, 1.5, n).cumsum() * 0.01
    proc = air + 10 + rng.normal(0, 0.5, n)
    rpm = np.clip(1500 + rng.normal(0, 150, n), 1200, 2800)
    torque = np.clip(40 + rng.normal(0, 8, n), 5, 80)
    wear = (np.arange(n) % 250)
    strain = (wear * torque) / 4000 + (proc - air) / 12
    fail = (strain + rng.normal(0, .2, n)) > np.quantile(strain, 0.97)
    return pd.DataFrame({"air_temp": air, "process_temp": proc, "rpm": rpm,
                         "torque": torque, "tool_wear": wear,
                         "failure": fail.astype(int)})

df = try_huggingface()
if df is None:
    df = try_uci()
if df is None:
    df = synthetic()
df = df[REQUIRED].reset_index(drop=True)
print(df.describe().round(2))


from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

FEATURES = ["air_temp", "process_temp", "rpm", "torque", "tool_wear"]
X, y = df[FEATURES].values, df["failure"].values
Xtr, Xte, ytr, yte, itr, ite = train_test_split(
    X, y, np.arange(len(df)), test_size=0.3, random_state=0, stratify=y)

clf = GradientBoostingClassifier(random_state=0)
clf.fit(Xtr, ytr)
proba_te = clf.predict_proba(Xte)[:, 1]
print(f"Failure prediction ROC-AUC: {roc_auc_score(yte, proba_te):.3f}")
print(classification_report(yte, proba_te > 0.5, digits=3))


df["fail_prob"] = clf.predict_proba(df[FEATURES].values)[:, 1]


timeline = df.iloc[::10][FEATURES + ["failure", "fail_prob"]].round(3)
timeline_json = timeline.to_dict(orient="list")

# Ranges for visual mapping (color scales, gauges)
ranges = {c: [float(df[c].min()), float(df[c].max())] for c in FEATURES}

html = r"""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>3D Digital Twin — Milling Machine</title>
<style>
  body { margin:0; background:#14161c; color:#e8e8e8; font-family:system-ui,sans-serif; overflow:hidden; }
  #hud { position:absolute; top:12px; left:12px; background:rgba(0,0,0,.55);
         padding:14px 18px; border-radius:10px; font-size:13px; line-height:1.7; min-width:250px;}
  #hud b { color:#7ec8ff; }
  #alert { font-weight:700; padding:2px 8px; border-radius:6px; }
  .ok   { background:#1e5c2f; } .warn { background:#7a5a12; } .bad { background:#8a1f1f; }
  #bar  { position:absolute; bottom:18px; left:12px; right:12px; display:flex; gap:10px; align-items:center;}
  #seek { flex:1; }
  button{ background:#2b3040; color:#e8e8e8; border:0; padding:8px 14px; border-radius:8px; cursor:pointer;}
  .gauge{ height:8px; width:220px; background:#333; border-radius:4px; margin:2px 0 8px; }
  .gauge>div{ height:100%; border-radius:4px; width:0%; }
</style>
</head>
<body>
<div id="hud">
  <div style="font-size:15px;font-weight:700;margin-bottom:6px">Milling Machine — Digital Twin</div>
  <div>step <b id="v_step">0</b> / __N__</div>
  <div>process temp <b id="v_pt">–</b> K</div>
  <div class="gauge"><div id="g_pt" style="background:#ff9e4f"></div></div>
  <div>spindle speed <b id="v_rpm">–</b> rpm</div>
  <div class="gauge"><div id="g_rpm" style="background:#7ec8ff"></div></div>
  <div>torque <b id="v_tq">–</b> Nm</div>
  <div class="gauge"><div id="g_tq" style="background:#b78cff"></div></div>
  <div>tool wear <b id="v_tw">–</b> min</div>
  <div class="gauge"><div id="g_tw" style="background:#ffd35c"></div></div>
  <div style="margin-top:6px">predicted failure risk: <span id="alert" class="ok">–</span></div>
  <div style="font-size:11px;opacity:.6;margin-top:4px">drag to orbit · replayed sensor feed</div>
</div>
<div id="bar">
  <button id="play">⏸ pause</button>
  <input type="range" id="seek" min="0" max="__NMAX__" value="0">
  <span style="font-size:12px">
    <button class="spd" data-s="1">1×</button>
    <button class="spd" data-s="4">4×</button>
    <button class="spd" data-s="16">16×</button>
  </span>
</div>
<script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
<script>
const DATA = __TIMELINE__;
const RANGES = __RANGES__;
const N = DATA.rpm.length;

// ---------- scene ----------
const scene = new THREE.Scene();
scene.background = new THREE.Color(0x14161c);
const camera = new THREE.PerspectiveCamera(50, innerWidth/innerHeight, .1, 100);
let camAz = 0.9, camEl = 0.5, camR = 7.5;
const renderer = new THREE.WebGLRenderer({antialias:true});
renderer.setSize(innerWidth, innerHeight); document.body.appendChild(renderer.domElement);
scene.add(new THREE.AmbientLight(0xffffff, .45));
const key = new THREE.DirectionalLight(0xffffff, .9); key.position.set(5,8,4); scene.add(key);
scene.add(new THREE.GridHelper(14, 14, 0x3a3f4c, 0x262a33));

// ---------- procedural milling machine ----------
const mach = new THREE.Group(); scene.add(mach);
const mat = c => new THREE.MeshStandardMaterial({color:c, roughness:.55, metalness:.35});
const bodyMat = mat(0x8a93a6);            // shared body material — tinted by temperature
const base   = new THREE.Mesh(new THREE.BoxGeometry(3,.5,2),  bodyMat); base.position.y=.25;        mach.add(base);
const column = new THREE.Mesh(new THREE.BoxGeometry(.7,3,.9), bodyMat); column.position.set(-1.1,2,0); mach.add(column);
const head   = new THREE.Mesh(new THREE.BoxGeometry(1.8,.8,.9),bodyMat); head.position.set(0,3.1,0);  mach.add(head);
const table  = new THREE.Mesh(new THREE.BoxGeometry(2.2,.25,1.3), mat(0x5f6678)); table.position.set(.2,.9,0); mach.add(table);
// spindle assembly (rotates with rpm)
const spindle = new THREE.Group(); spindle.position.set(.3,2.7,0); mach.add(spindle);
const shaft = new THREE.Mesh(new THREE.CylinderGeometry(.12,.12,.8,24), mat(0xcfd6e4)); spindle.add(shaft);
const tool  = new THREE.Mesh(new THREE.CylinderGeometry(.05,.02,.5,16), mat(0xffd35c)); tool.position.y=-.6; spindle.add(tool);
const flange= new THREE.Mesh(new THREE.BoxGeometry(.5,.1,.1), mat(0xcfd6e4)); spindle.add(flange); // makes rotation visible
// warning beacon (lights up with predicted failure risk)
const beaconMat = new THREE.MeshStandardMaterial({color:0x113311, emissive:0x000000});
const beacon = new THREE.Mesh(new THREE.SphereGeometry(.14,16,16), beaconMat); beacon.position.set(-1.1,3.7,0); mach.add(beacon);

// ---------- helpers ----------
const lerp=(a,b,t)=>a+(b-a)*t;
const norm=(v,k)=>Math.min(1,Math.max(0,(v-RANGES[k][0])/(RANGES[k][1]-RANGES[k][0]+1e-9)));
const coldC=new THREE.Color(0x8a93a6), hotC=new THREE.Color(0xd2543a);

// ---------- playback state ----------
let idx=0, playing=true, speed=4, spin=0;
const seek=document.getElementById('seek');
document.getElementById('play').onclick=e=>{playing=!playing;e.target.textContent=playing?'⏸ pause':'▶ play';};
document.querySelectorAll('.spd').forEach(b=>b.onclick=()=>speed=+b.dataset.s);
seek.oninput=e=>{idx=+e.target.value;};

// simple orbit controls (drag)
let drag=false,px=0,py=0;
addEventListener('pointerdown',e=>{if(e.target.tagName==='CANVAS'){drag=true;px=e.clientX;py=e.clientY;}});
addEventListener('pointerup',()=>drag=false);
addEventListener('pointermove',e=>{if(drag){camAz+=(e.clientX-px)*.005;camEl=Math.min(1.4,Math.max(.05,camEl+(e.clientY-py)*.005));px=e.clientX;py=e.clientY;}});
addEventListener('resize',()=>{camera.aspect=innerWidth/innerHeight;camera.updateProjectionMatrix();renderer.setSize(innerWidth,innerHeight);});

// ---------- sync loop: sensor state -> twin state ----------
function applyState(i){
  const pt=DATA.process_temp[i], rpm=DATA.rpm[i], tq=DATA.torque[i],
        tw=DATA.tool_wear[i], p=DATA.fail_prob[i];
  // temperature -> body tint
  bodyMat.color.copy(coldC).lerp(hotC, norm(pt,'process_temp'));
  // torque -> slight head "load" tilt
  head.rotation.z = -0.06 * norm(tq,'torque');
  // tool wear -> tool shortens & darkens
  tool.scale.y = 1 - 0.55*norm(tw,'tool_wear');
  tool.material.color.setHSL(0.13, 0.9, lerp(0.62, 0.25, norm(tw,'tool_wear')));
  // failure risk -> beacon
  const lvl = p>0.5 ? 2 : (p>0.2 ? 1 : 0);
  beaconMat.emissive.setHex([0x001a00,0x8a6a00,0xaa1111][lvl]);
  beaconMat.color.setHex([0x113311,0x554400,0x662222][lvl]);
  // HUD
  const set=(id,v)=>document.getElementById(id).textContent=v;
  set('v_step',i); set('v_pt',pt.toFixed(1)); set('v_rpm',rpm.toFixed(0));
  set('v_tq',tq.toFixed(1)); set('v_tw',tw.toFixed(0));
  const g=(id,k,v)=>document.getElementById(id).style.width=(100*norm(v,k))+'%';
  g('g_pt','process_temp',pt); g('g_rpm','rpm',rpm); g('g_tq','torque',tq); g('g_tw','tool_wear',tw);
  const al=document.getElementById('alert');
  al.textContent=(100*p).toFixed(1)+'%';
  al.className=['ok','warn','bad'][lvl];
  seek.value=i;
  return rpm;
}

let acc=0;
function animate(){
  requestAnimationFrame(animate);
  camera.position.set(camR*Math.cos(camEl)*Math.cos(camAz), camR*Math.sin(camEl)+1,
                      camR*Math.cos(camEl)*Math.sin(camAz));
  camera.lookAt(0,1.4,0);
  if(playing){ acc+=speed; while(acc>=8){ acc-=8; idx=(idx+1)%N; } }
  const rpm=applyState(idx);
  spin += (rpm/60) * 0.05;          // spindle visually rotates with live rpm
  spindle.rotation.y = spin;
  renderer.render(scene,camera);
}
animate();
</script>
</body></html>"""

html = (html
        .replace("__TIMELINE__", json.dumps(timeline_json))
        .replace("__RANGES__", json.dumps(ranges))
        .replace("__NMAX__", str(len(timeline) - 1))
        .replace("__N__", str(len(timeline) - 1)))

out_path = os.path.join(OUT, "digital_twin.html")
with open(out_path, "w") as f:
    f.write(html)
print(f"Twin exported: {out_path} ({os.path.getsize(out_path)/1e6:.2f} MB)")

from IPython.display import IFrame, display
display(IFrame(src="digital_twin.html", width=1000, height=650))

  HF candidate pauhidalgoo/ai4i2020 failed: DatasetNotFoundError
  HF candidate mstz/machine_failure failed: DatasetNotFoundError
  HF candidate Chances/ai4i2020 failed: DatasetNotFoundError
Loaded from UCI (10000 rows)
       air_temp  process_temp       rpm    torque  tool_wear   failure
count   10000.0      10000.00  10000.00  10000.00   10000.00  10000.00
mean      300.0        310.01   1538.78     39.99     107.95      0.03
std         2.0          1.48    179.28      9.97      63.65      0.18
min       295.3        305.70   1168.00      3.80       0.00      0.00
25%       298.3        308.80   1423.00     33.20      53.00      0.00
50%       300.1        310.10   1503.00     40.10     108.00      0.00
75%       301.5        311.10   1612.00     46.80     162.00      0.00
max       304.5        313.80   2886.00     76.60     253.00      1.00
Failure prediction ROC-AUC: 0.968
              precision    recall  f1-score   support

           0      0.986     0.998     0.992      289